In [41]:
from openai import OpenAI
from dotenv import load_dotenv

import os
import json
from IPython.display import Markdown
import chromadb

load_dotenv(override=True)

True

In [42]:
# chroma_client = chromadb.Client()
chroma_client = chromadb.PersistentClient(path="./")
# collection = chroma_client.create_collection(name="restaurant_vector_db")

In [43]:
llm_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("OPEN_ROUTER_KEY"),
)

MODEL = "openai/gpt-oss-120b"      

In [44]:
menu = [
    {"name": "Zinger Burger", "price": 850},
    {"name": "Chicken Biryani", "price": 450},
    {"name": "Club Sandwich", "price": 700}
]

complaint_list = []

order_list = [
    "OR-5555", "OR-6666", "OR-7777", "OR-8888", "OR-9999"
]

In [45]:
def fetch_menu():
    return menu

def log_complaint(customer_name, order_id, phone, complaint):
    temp_dict = {
        "customer_name": customer_name,
        "order_id": order_id,
        "phone": phone,
        "complaint": complaint,
    }
    complaint_list.append(temp_dict)
    return f'Total Complaint: {len(complaint_list)}'

def validate_order(order_id):
    if order_id in order_list:
        return True
    else:
        return False

tools = [
    {
        "type": "function",
        "function": {
                "name": "fetch_menu",
                "description": "Get the restaurant menu",
                "parameters": {"type": "object", "properties": {}},
        },
    },
    {
    "type": "function",
    "function": {
        "strict": True,
        "name": "log_complaint",
        "description": "Log the complaint",
        "parameters": {
            "type": "object",
            "properties": {
                "customer_name": {"type": "string", "description": "Customer Name who is logging complaint."},
                "order_id": {"type": "string", "description": "Order ID like OR-0000"},
                "phone": {"type": "string", "description": "Customer Phone Number or '-'"},
                "complaint": {"type": "string", "description": "Customer complaint, not more than 8 words."},
            },
            "required": ["customer_name", "order_id", "phone", "complaint"],
            "additionalProperties": False,
        },
    },
},
    {
        "type": "function",
        "function": {
                "name": "validate_order",
                "description": "Validate the Order",
                "parameters": {"type": "object", "properties": {
                    "order_id":{
                        "Type":"string",
                        "description":"The Order ID of which the cutomer ordered food, Order ID sgould be OR-0000"
                    }
                },
                "required": ["order_id"],
            "additionalProperties": False,
            },
        },
    },
]

In [46]:
messages = [
    {
      "role": "system",  
      "content": """
       You are the AI Customer Support Assistant for **Eatery Restaurant**. Your job is to answer customer queries in a friendly, professional, and helpful way.

        ### Tools
        1. **`get_menu`** — Call whenever customers ask about menu items, prices, ingredients, or food recommendations. Use the tool's response to provide accurate information, The currency icon must be in Pakistani currency PKR.
        2. **`log_complaint`** — Call whenever a customer raises a complaint. Collect the necessary details and log the complaint using this schema:
        3. **`validate_order`** — Call whenever customers enter an Order Id, call this function and validate that order Id if it matches order_list, if it matches then execute the desired statement or task that the user wants but if the order Id doesn't match then say it is invalid and ask the user to enter the correct one.

        ```python
        complaint = {
            "customer_name": "",
            "phone": "",
            "complaint": "",
            "order_id": "",
            "status": "Pending"
        }
        ```

        Ask for missing necessary details before logging. The `order_id` can be empty if unavailable. Set the initial status to `"Pending"`. Confirm the complaint is logged only after the tool succeeds.

        ### Guidelines
        - Keep replies short, natural, and friendly.
        - Never invent menu details, prices, or restaurant policies.
        - If you don't know something, politely say so.
        - Make every customer feel valued.
      """
    }
]


In [47]:
function_names = {
    "fetch_menu": fetch_menu,
    "log_complaint": log_complaint,
    "validate_order": validate_order,
}



def ask_assistant(question):

    messages.append({
        "role": "user", 
        "content": question
    })

    completion = llm_client.chat.completions.create(
        model=MODEL, 
        messages=messages, 
        tools=tools,
        temperature=0
    )
   
    reply = completion.choices[0].message

    print("LLM REPY", reply)

    if reply.tool_calls:                       
        messages.append(reply)

        for call in reply.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            result = function_names[call.function.name](**args)     
            print("Tool called:", call.function.name, args)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

        completion = llm_client.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    return Markdown(completion.choices[0].message.content)

In [48]:
# ask_assistant("Tell me the Menu?")

In [49]:
ask_assistant("I ordered Burger but sandwich was served and the bill was made with burger, My name is Abdul Salam, OR-1778, phone is 03269222700")

LLM REPY ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_9798a6ff-f44f-4f0a-bd12-1547691bfe59', function=Function(arguments='{"complaint":"Wrong item served","customer_name":"Abdul Salam","order_id":"OR-1778","phone":"03269222700"}', name='log_complaint'), type='function')], reasoning='We need to log a complaint. Need to collect complaint description <=8 words. The user says: "I ordered Burger but sandwich was served and the bill was made with burger". That\'s the complaint. Need to ensure complaint <=8 words. Let\'s count: "Wrong item served" is short. We can ask for a concise complaint? The user already gave details. We can summarize: "Wrong item served". That\'s 3 words. Use that as complaint. Use log_complaint tool with fields: complaint, customer_name, order_id, phone.\n\nProceed to call log_complaint.')
Tool called: log_complaint {'complaint': 'Wrong item

Your complaint has been logged successfully, Abdul Salam. We’ll look into the mix‑up with your order (OR‑1778) right away. Thank you for letting us know, and we’ll get back to you shortly.

In [50]:
# ask_assistant("My order ID is OR-5555, validate my order, is it a valid order id? Check from order_lists")

In [51]:
# ask_assistant("My order ID is OR-1011, validate my order, is it a valid order id? Check from order_lists")

In [52]:
# print(complaint_list)

In [53]:
# ask_assistant("What is the restaurant's name")

In [54]:
# ask_assistant("Why is your restaurant name Salman Pasta?")